In [13]:
from transformers import ElectraTokenizer, ElectraForSequenceClassification
import torch
import pandas as pd

In [14]:
# Load model and tokenizer from TRAINING folder
model_path = "TRAINING\MODELS"

In [15]:
tokenizer = ElectraTokenizer.from_pretrained(model_path)
model = ElectraForSequenceClassification.from_pretrained(model_path, num_labels=2)

In [16]:
# Function to predict
def predict_suicidal(text: str):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs).item()
        suicide_score = probs[0][1].item()
    
    label = "suicide" if pred == 1 else "non-suicide"
    return label, suicide_score

In [17]:
def prepare_chat_data(file_path):
    """
    Reads a .txt file and returns all its content as a single cleaned text string.
    No saving or printing — just returns the processed text.
    """

    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            # Read all lines and remove extra spaces or newlines
            text = " ".join(line.strip() for line in file if line.strip())

        return text

    except FileNotFoundError:
        print(f"❌ File not found: {file_path}")
        return ""
    except Exception as e:
        print(f"❌ Error reading file: {e}")
        return ""


In [18]:
# Example usage
text = prepare_chat_data("C:\Projects\SUICIDE_PROJECT\SUICIDE_PROJECT\SUICIDE_PROJECT\Texts\sucide3.txt")
print("Given input:",text)
prediction, score = predict_suicidal(text)
text_prediction=prediction
text_probability_score =round(score,4)


Given input: User A: I’ve been feeling a bit low lately because of exams. User B: Yeah, same here. Too much pressure. User A: I just need a break and maybe some sleep. User B: Totally, let’s chill this weekend.


In [19]:
import speech_recognition as sr

def audio_file_to_text(file_path):
    recognizer = sr.Recognizer()

    try:
        # Load the audio file
        with sr.AudioFile(file_path) as source:
            audio_data = recognizer.record(source)

        # Convert speech to text using Google API
        text = recognizer.recognize_google(audio_data)
        return text

    except sr.UnknownValueError:
        return "❌ Could not understand the audio"
    except sr.RequestError as e:
        return f"❌ API request error: {e}"
    except Exception as e:
        return f"❌ Error: {e}"



In [20]:

text = audio_file_to_text(r"C:\Projects\SUICIDE_PROJECT\SUICIDE_PROJECT\SUICIDE_PROJECT\Audios\sucide audio 2.wav")
print("Given input:",text)
prediction, score = predict_suicidal(text)
voice_prediction=prediction
voice_probability_score =round(score,4)

Given input: sometimes I wonder if life would be better without me arrow I feel like a burden to everyone and I just can't seem to do anything right it's hard to find a reason to keep going when everyday feels like a struggle I don't know if anyone would even notice if I disappear I just feel so alone trapped in my own thought it's exhausting pretending to be ok when I am breaking inside have been thinking about how nice should be it would be just sleep and never wake up it's not like anyone I wish things were different but it feels like nothing ever changes I keep telling myself to hold on but I don't know how much longer I can sometime I just want it all to stop


In [28]:
from deepface import DeepFace
import cv2
from collections import Counter
import os

def analyze_emotion_video(video_path, output_path="output_emotion_video.mp4"):
    if not os.path.exists(video_path):
        print("❌ Video file not found!")
        return

    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    print(f"✅ Processing started...\nFPS: {fps}, Resolution: {width}x{height}")
    frame_count = 0
    emotions_detected = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_count += 1

        if frame_count % 5 != 0:  # process every 5th frame for speed
            out.write(frame)
            continue

        try:
            result = DeepFace.analyze(frame, actions=['emotion'], enforce_detection=False)
            dominant_emotion = result[0]['dominant_emotion']
            emotions_detected.append(dominant_emotion)

            cv2.putText(frame, f"{dominant_emotion}", (30, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)
        except Exception as e:
            print("Error:", e)

        cv2.imshow("Emotion Detection (DeepFace)", frame)
        out.write(frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    out.release()
    cv2.destroyAllWindows()

    if emotions_detected:
        dominant = Counter(emotions_detected).most_common(1)[0][0]
        return dominant
    else:
        return None


video_path = r"C:\Projects\SUICIDE_PROJECT\SUICIDE_PROJECT\SUICIDE_PROJECT\Videos\3712705-uhd_4096_2160_25fps.mp4"
video_result=analyze_emotion_video(video_path)


✅ Processing started...
FPS: 25, Resolution: 4096x2160


In [31]:
suicidal_emotions = {"sad", "angry", "fear", "disgust"}
non_suicidal_emotions = {"neutral", "happy", "surprise"}
label=""
score=0.0
if video_result in suicidal_emotions:
    label = "Suicidal"
    score = 0.8  # assign stronger weight for negative state
elif video_result in non_suicidal_emotions:
    label = "Non-Suicidal"
    score = 0.3
else:
    label = "Non-Suicidal"
    score = 0.5


In [34]:
def fuse_predictions(text_output=None, voice_output=None, video_output=None):
    

    # Base weights
    weights = {
        "text": 0.4 if text_output else 0.0,
        "voice": 0.4 if voice_output else 0.0,
        "video": 0.2 if video_output else 0.0
    }

    # Normalize weights (so total = 1.0)
    total_weight = sum(weights.values())
    if total_weight == 0:
        print("❌ No model outputs provided.")
        return "Unknown", 0.0
    for key in weights:
        weights[key] = weights[key] / total_weight

    # Extract probabilities
    text_prob = text_output[1] if text_output else None
    voice_prob = voice_output[1] if voice_output else None
    video_prob = video_output[1] if video_output else None

    # Weighted probability fusion
    final_prob = 0.0
    if text_prob is not None:
        final_prob += text_prob * weights["text"]
    if voice_prob is not None:
        final_prob += voice_prob * weights["voice"]
    if video_prob is not None:
        final_prob += video_prob * weights["video"]

    # Final decision threshold
    final_label = "Suicidal" if final_prob >= 0.5 else "Non-Suicidal"


    return final_label, final_prob


In [38]:
text_out = (text_prediction,text_probability_score)
voice_out = (voice_prediction,voice_probability_score)
video_out = (label,score)
label,score = fuse_predictions(text_out, voice_out, video_out)
print("Final prediction:",label)
print("sucidal Probability :",round(score,2)*100)


Final prediction: Non-Suicidal
sucidal Probability : 50.0
